# Dashboard de Equidade de Gênero em TI

**Implementação interativa do protocolo padronizado (Natal, 2026).**

Este notebook calcula passo a passo as **três métricas** do protocolo (M1, M2, M3), identifica automaticamente a **fase crítica** do pipeline e gera as visualizações finais. Ele foi projetado para funcionar com dados de **qualquer instituição de ensino superior** — basta colocar os arquivos na pasta e executar as células.

---

## Antes de começar — prepare sua pasta no Google Drive

**Siga exatamente estes passos antes de executar qualquer célula:**

1. No Google Drive, crie uma pasta chamada **`equidade_ti`** (ou outro nome, mas anote).
2. Coloque **este notebook** (`dashboard_equidade_ti.ipynb`) dentro dessa pasta.
3. Coloque os **arquivos de dados da sua instituição** na mesma pasta.

O notebook detecta automaticamente o tipo de cada arquivo pelo conteúdo — não pelo nome — e normaliza as colunas independente de como a sua instituição nomeou cada campo:

| Tipo detectado | O que o arquivo precisa ter | Métricas geradas |
|---|---|---|
| **COORTE** | Coluna de gênero + ano de ingresso + coluna de situação (status, situação, situacao_vinculo…) com valores como ATIVO, CANCELADO, CONCLUÍDO | **M1 + M2 + M3** |
| **CONCLUSÃO** | Coluna de gênero + ano de conclusão (sem coluna de situação) | M3 |
| **INGRESSO** | Coluna de gênero + ano de ingresso (sem coluna de situação) | M1 |

Formatos aceitos: **CSV** (qualquer separador e encoding) · **Excel `.xlsx`** · **JSON**.

> **O notebook interpreta automaticamente nomes de colunas diferentes.** Por exemplo, a coluna de gênero pode se chamar `sexo`, `tp_sexo`, `genero`, `gen_aluno` — o notebook normaliza tudo para o mesmo campo interno. Veja a lista completa de sinônimos na seção 3.

4. Se o nome da sua pasta for diferente de `equidade_ti`, edite a variável `NOME_PASTA_DRIVE` na **célula 1 (Setup)** antes de executar.

**Estrutura esperada no Drive:**
```
Minha Unidade/
└── equidade_ti/ ← pasta criada por você
 ├── dashboard_equidade_ti.ipynb ← este notebook
 ├── discentes-2019.csv ← arquivo(s) da sua instituição
 ├── discentes-egressos-2023.csv ← (opcional)
 └── ...
```

---

## O que o protocolo mede

| Métrica | Fórmula | O que captura |
|---|---|---|
| **M1** | (Mulheres Ingressantes / Total Ingressantes) × 100 | Equidade na **entrada** |
| **M2** | TEF − TEM, onde TEF = (Mulheres Evadidas / Mulheres Ingressantes) × 100 e TEM análogo para homens | Gap de **evasão por gênero** |
| **M3** | (Mulheres Concluintes no ano / Total Concluintes no ano) × 100 | Equidade na **saída** |

**Classificação de M1 (Tabela 1 do protocolo):**

- `< 20%` → Disparidade Crítica
- `20–30%` → Disparidade Significativa
- `30–40%` → Desbalanceamento Moderado
- `40–50%` → Aproximação da Paridade
- `≥ 50%` → Paridade Alcançada

**Identificação de fase crítica (Tabela 2):** Ingresso (M1<30%) · Permanência (M2>5 p.p.) · Conclusão (M3<M1−5).
A maior gravidade vira a fase crítica e o relatório mostra recomendações específicas para ela.

---

## Quais arquivos colocar na pasta

| Tipo | Como reconhecer | O que gera |
|---|---|---|
| **INGRESSO** | só ingressantes; ex.: `chamada_regular_sisu_*.csv` | **M1** |
| **CONCLUSÃO** | só concluintes; ex.: `discentes-egressos-AAAA.csv` (UFRN) | **M3** |
| **COORTE** | ingressantes + coluna `status` (`CANCELADO`/`CONCLUÍDO`/…); ex.: `discentes-AAAA.csv` | **M1 + M2 + M3** |

Formatos aceitos: **CSV** (qualquer separador/encoding) · **Excel `.xlsx`** · **JSON**.

## 1. Setup

Monta o Google Drive, define a pasta de dados e instala as bibliotecas necessárias.

> **Edite `NOME_PASTA_DRIVE`** se o nome da sua pasta no Drive for diferente de `equidade_ti`.

In [2]:
import os, sys, re, json, glob, unicodedata, subprocess

# ── Instala dependências ──────────────────────────────────────────────────
for pkg in ["pandas", "plotly", "panel", "openpyxl"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import pandas as pd
import plotly.graph_objects as go
import panel as pn
pn.extension("plotly", sizing_mode="stretch_width")

CORES = {
    "f": "#E91E63", "m": "#1976D2", "verde": "#4CAF50",
    "amarelo": "#FF9800", "vermelho": "#F44336", "roxo": "#9C27B0",
}

# ── Configuração da pasta ─────────────────────────────────────────────────
# Mude aqui se o nome da sua pasta no Drive for diferente.
NOME_PASTA_DRIVE = "equidade_ti"

# Monta o Google Drive (necessário no Colab; ignorado fora dele)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PASTA = f"/content/drive/MyDrive/{NOME_PASTA_DRIVE}"
    print(f" Google Drive montado. Pasta de dados: {PASTA}")
except ImportError:
    # Fora do Colab: usa o diretório atual do notebook
    PASTA = os.getcwd()
    print(f"ℹ Colab não detectado. Pasta de dados: {PASTA}")

if not os.path.isdir(PASTA):
    print(f"\nATENÇÃO: a pasta '{PASTA}' não existe no Drive.")
    print(f" Crie a pasta '{NOME_PASTA_DRIVE}' em Minha Unidade e coloque os arquivos lá.")
else:
    arquivos = os.listdir(PASTA)
    print(f" Arquivos encontrados ({len(arquivos)}): {arquivos[:10]}")

/tmp/ipykernel_9277/3502020443.py:12: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension("plotly", sizing_mode="stretch_width")


Mounted at /content/drive
 Google Drive montado. Pasta de dados: /content/drive/MyDrive/equidade_ti
 Arquivos encontrados (7): ['discentes-2018.csv', 'discentes-2017.csv', 'discentes-egressos-2025.csv', 'discentes-egressos-2024.csv', 'discentes-egressos-2023.csv', 'discentes-2019.csv', 'dashboard_equidade_ti.ipynb']


## 2. Configurar os termos de classificação de TI

A classificação automática usa duas listas:

- **INCLUIR** — termos que indicam curso de TI (basta um casar).
- **EXCLUIR** — termos que ignorar o curso, *exceto* se também casar algum INCLUIR. Isso evita falsos positivos (ex.: `engenharia elétrica` não é TI, mas `engenharia de computação` é).

Para adicionar um curso novo (ex.: *Engenharia Biomédica*), basta colocar `biomedica` na lista INCLUIR e reexecutar a célula. Para remover um falso positivo, adicione em EXCLUIR.

In [6]:
CONFIG_TI = {
    "incluir": [
        "computac", "computaç", "informatic", "informátic", "software",
        "sistemas de informac", "sistemas de informaç",
        "tecnologia da informac", "tecnologia da informaç",
        "ciencia de dado", "ciência de dado", "data science",
        "inteligencia artificial", "inteligência artificial",
        "analise e desenvolvimento", "análise e desenvolvimento",
        "analise de sistema", "análise de sistema",
        "redes de computador", "seguranca da informac", "segurança da informaç",
        "banco de dado", "jogos digitais", "telecomunicac", "telecomunicaç",
        "mecatronic", "mecatrônic",
        # >>> adicione aqui termos novos a serem considerados TI <<<
    ],
    "excluir": [
        "eletric", "elétric", "civil", "mecanic", "mecânic", "quimic", "químic",
        "ambiental", "producao", "produção", "aliment", "medicin", "direito",
        "pedagog", "letras", "histori", "história", "psicolog", "biolog",
        "farmac", "fármac", "odontolog", "veterinar", "nutric", "nutriç",
        "enferm", "contab", "contáb", "administrac", "administraç", "econom",
        "educacao fisica", "educação física", "arquitetura", "jornalis",
        "publicidad", "sociolog", "filosof", "geograf", "matemat", "matemát",
        "estatistic", "estatístic", "florestal", "agronom", "zootecn",
        "music", "músic", "artes", "teatro", "dança", "danca",
        # >>> adicione aqui termos que devem ignorar o curso <<<
    ],
}

def eh_ti(nome):
    """True se `nome` parece ser de um curso de TI."""
    n = str(nome).lower()
    for e in CONFIG_TI["excluir"]:
        if e in n:
            for t in CONFIG_TI["incluir"]:
                if t in n:
                    return True
            return False
    for t in CONFIG_TI["incluir"]:
        if t in n:
            return True
    return False

print(f"Termos INCLUIR ({len(CONFIG_TI['incluir'])}):",
      ", ".join(CONFIG_TI["incluir"][:6]), "...")
print(f"Termos EXCLUIR ({len(CONFIG_TI['excluir'])}):",
      ", ".join(CONFIG_TI["excluir"][:6]), "...")
print("\nTeste do classificador eh_ti():")
for t in ["CIÊNCIA DA COMPUTAÇÃO", "ENGENHARIA ELÉTRICA",
          "ENGENHARIA DE COMPUTAÇÃO", "DIREITO",
          "ANÁLISE E DESENVOLVIMENTO DE SISTEMAS"]:
    print(f" {t}: {eh_ti(t)}")

Termos INCLUIR (27): computac, computaç, informatic, informátic, software, sistemas de informac ...
Termos EXCLUIR (52): eletric, elétric, civil, mecanic, mecânic, quimic ...

Teste do classificador eh_ti():
 CIÊNCIA DA COMPUTAÇÃO: True
 ENGENHARIA ELÉTRICA: False
 ENGENHARIA DE COMPUTAÇÃO: True
 DIREITO: False
 ANÁLISE E DESENVOLVIMENTO DE SISTEMAS: True


## 3. Leitura genérica e normalização de colunas

Cada instituição exporta dados com nomes de colunas diferentes. O notebook resolve isso em três etapas automáticas:

**`ler_arquivo_qualquer`** — abre CSV, Excel ou JSON sem que você precise configurar separador ou encoding: o código testa as combinações mais comuns e usa a que funcionar.

**`normalizar_colunas`** — traduz qualquer nome de coluna para um **schema interno**, usando a tabela de sinônimos abaixo. Se a sua planilha usar um nome não listado, basta adicioná-lo na variável `SINONIMOS_COLUNAS`.

| Campo canônico | Nomes aceitos automaticamente |
|---|---|
| `genero` | `sexo`, `tp_sexo`, `genero`, `gen_aluno` |
| `ano_ingresso` | `ano_ingresso`, `ano_de_ingresso`, `ingresso_ano` |
| `ano_conclusao` | `ano_conclusao`, `ano_de_conclusao`, `conclusao_ano` |
| `curso` | `nome_curso`, `no_curso`, `curso`, `ds_curso` |
| `status` | `status`, `situacao`, `situacao_vinculo`, `status_aluno`, `situacao_aluno`, `status_discente` |
| `nome_ies` | `nome_ies`, `no_ies`, `ies` |
| `sigla_ies` | `sigla_ies`, `sg_ies` |
| `nivel_ensino` | `nivel_ensino`, `ds_nivel_ensino` |
| `matricula` | `matricula`, `id_matricula`, `nu_matricula` |

**`detectar_tipo_arquivo`** — depois de normalizar as colunas, decide se o arquivo é **COORTE**, **CONCLUSÃO** ou **INGRESSO** com base no que encontrar:
- Tem `status` com valores além de "concluído" (ex.: ATIVO, CANCELADO) → **COORTE** (gera M1+M2+M3)
- Tem `ano_conclusao` mas sem status variado → **CONCLUSÃO** (gera M3)
- Caso contrário → **INGRESSO** (gera M1)

In [8]:
def _slug(s):
    """Normaliza string para comparação (lowercase + sem acentos + sem ruído)."""
    if s is None: return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip().strip('"').replace("-", "_").replace(" ", "_")
    return re.sub(r"_+", "_", s)


def _detectar_sep(caminho):
    """Heurística simples para encoding + separador de um CSV."""
    for enc in ["utf-8", "latin-1"]:
        try:
            with open(caminho, "r", encoding=enc) as f:
                header = f.readline()
                for sep in ["|", ";", ",", "\t"]:
                    if header.count(sep) >= 5:
                        return enc, sep
        except Exception:
            continue
    return "latin-1", ";"


def ler_arquivo_qualquer(caminho):
    """Lê CSV/XLSX/JSON e devolve DataFrame (tudo como str)."""
    ext = os.path.splitext(caminho)[1].lower()
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(caminho, dtype=str)
    if ext == ".json":
        with open(caminho, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, dict) and "data" in data: data = data["data"]
            return pd.DataFrame(data).astype(str)
    enc, sep = _detectar_sep(caminho)
    return pd.read_csv(caminho, encoding=enc, sep=sep, dtype=str, low_memory=False)


SINONIMOS_COLUNAS = {
    "ano": ["ano", "nu_ano", "ano_referencia"],
    "ano_ingresso": ["ano_ingresso", "ano_de_ingresso", "ingresso_ano"],
    "ano_conclusao": ["ano_conclusao", "ano_de_conclusao", "conclusao_ano"],
    "genero": ["sexo", "tp_sexo", "genero"],
    "curso": ["nome_curso", "no_curso", "curso", "ds_curso"],
    "nome_ies": ["nome_ies", "no_ies", "ies"],
    "sigla_ies": ["sigla_ies", "sg_ies"],
    "uf_ies": ["uf_ies", "sg_uf_ies", "uf"],
    "turno": ["turno", "ds_turno"],
    "aprovado": ["aprovado", "st_aprovado", "st_matricula"],
    "status": ["status", "situacao", "situacao_vinculo", "status_aluno",
               "situacao_aluno", "status_discente"],
    "forma_ingresso": ["forma_ingresso", "ds_forma_ingresso"],
    "nivel_ensino": ["nivel_ensino", "ds_nivel_ensino"],
    "nivel_ensino_sigla": ["sigla_nivel_ensino"],
    "matricula": ["matricula", "id_matricula", "nu_matricula"],
    "cpf": ["cpf", "nu_cpf"],
}

def normalizar_colunas(df):
    """Renomeia colunas do `df` para o schema canônico via SINONIMOS_COLUNAS."""
    rename = {}
    for col in df.columns:
        sc = _slug(col)
        nova = sc
        for canon, lista in SINONIMOS_COLUNAS.items():
            if sc in [_slug(x) for x in lista]:
                nova = canon
                break
        rename[col] = nova
    df2 = df.rename(columns=rename)
    df2 = df2.loc[:, ~df2.columns.duplicated()]
    return df2


STATUS_MAP = {
    "CONCLUIDO": "CONCLUINTE", "CONCLUÍDO": "CONCLUINTE",
    "FORMADO": "CONCLUINTE", "DEFENDIDO": "CONCLUINTE",
    "CONCLUSAO": "CONCLUINTE", "GRADUADO": "CONCLUINTE",
    "CANCELADO": "EVADIDO", "JUBILADO": "EVADIDO",
    "DESLIGADO": "EVADIDO", "EVADIDO": "EVADIDO",
    "EXCLUIDO": "EVADIDO", "EXCLUÍDO": "EVADIDO",
    "ABANDONO": "EVADIDO", "TRANSFERIDO":"EVADIDO",
    "TRANSF. COMPULSORIA": "EVADIDO",
    "ATIVO": "ATIVO", "ATIVO - FORMANDO": "ATIVO",
    "CADASTRADO": "ATIVO", "MATRICULADO": "ATIVO",
    "TRANCADO": "ATIVO",
}

def normalizar_status(serie):
    """Mapeia valores brutos de status para 4 categorias canônicas.
    Valores desconhecidos viram 'ATIVO' (protocolo §4.1)."""
    s = serie.astype(str).str.strip().str.upper()
    return s.map(STATUS_MAP).fillna("ATIVO")


def detectar_tipo_arquivo(df):
    """Retorna 'COORTE' | 'CONCLUSAO' | 'INGRESSO'."""
    cols = set(df.columns)
    if "status" in cols:
        cats = set(normalizar_status(df["status"]).unique())
        if cats - {"CONCLUINTE"}:
            return "COORTE"
    if "ano_conclusao" in cols and df["ano_conclusao"].notna().any():
        return "CONCLUSAO"
    return "INGRESSO"

print("Funções utilitárias carregadas.")

Funções utilitárias carregadas.


## 4. Carregar e classificar os arquivos da pasta

Lista todos os arquivos da pasta do Drive e detecta automaticamente o tipo de cada um. O nome do arquivo não importa — a detecção é feita pelo **conteúdo** (quais colunas existem e quais valores o campo de situação tem).

> Se um arquivo aparecer como tipo errado, verifique se a coluna de situação tem o nome esperado (ex.: `status`, `situacao`). Caso use um nome diferente, adicione-o em `SINONIMOS_COLUNAS` na célula anterior.

In [10]:
# Busca todos os arquivos de dados dentro da pasta do Drive
PADROES = [
    os.path.join(PASTA, "discentes-*.csv"),
    os.path.join(PASTA, "discentes-*.xlsx"),
    os.path.join(PASTA, "discentes-*.json"),
    os.path.join(PASTA, "discentes_*.csv"),
    os.path.join(PASTA, "chamada_regular*.csv"),
    os.path.join(PASTA, "sisu_ti_consolidado.csv"),
    os.path.join(PASTA, "ufrn_ti_consolidado.csv"),
]
CANDIDATOS = []
for p in PADROES:
    CANDIDATOS.extend(glob.glob(p))

if not CANDIDATOS:
    print(" Nenhum arquivo encontrado em:", PASTA)
    print(" Verifique se a pasta existe no Drive e se os arquivos foram colocados lá.")
else:
    print("Arquivos encontrados:")
    for f in CANDIDATOS:
        print(f" - {os.path.basename(f)}")

    print("\nTipo detectado por arquivo:")
    for f in CANDIDATOS:
        try:
            amostra = ler_arquivo_qualquer(f).head(200)
            amostra = normalizar_colunas(amostra)
            tipo = detectar_tipo_arquivo(amostra)
            print(f" {os.path.basename(f):45s} → {tipo}")
        except Exception as e:
            print(f" {os.path.basename(f):45s} → ERRO: {e}")

Arquivos encontrados:
 - discentes-2018.csv
 - discentes-2017.csv
 - discentes-egressos-2025.csv
 - discentes-egressos-2024.csv
 - discentes-egressos-2023.csv
 - discentes-2019.csv

Tipo detectado por arquivo:
 discentes-2018.csv                            → COORTE
 discentes-2017.csv                            → COORTE
 discentes-egressos-2025.csv                   → CONCLUSAO
 discentes-egressos-2024.csv                   → CONCLUSAO
 discentes-egressos-2023.csv                   → CONCLUSAO
 discentes-2019.csv                            → COORTE


## 5. Carregar os arquivos de COORTE (M1 + M2 + M3 da mesma fonte)

Arquivos com ingressantes **e** situação acadêmica permitem calcular as três métricas do protocolo a partir de uma única fonte — independente da instituição de origem. O campo de situação pode ter qualquer nome (`status`, `situacao`, `situacao_vinculo`, etc.) e qualquer conjunto de valores; o notebook normaliza tudo para 4 categorias canônicas:

| Categoria canônica | Valores brutos reconhecidos automaticamente |
|---|---|
| **CONCLUINTE** | `CONCLUÍDO`, `FORMADO`, `DEFENDIDO`, `GRADUADO`, `CONCLUSAO` e variações |
| **EVADIDO** | `CANCELADO`, `JUBILADO`, `DESLIGADO`, `EVADIDO`, `EXCLUÍDO`, `ABANDONO`, `TRANSFERIDO`, `TRANSF. COMPULSORIA` e variações |
| **ATIVO** | `ATIVO`, `ATIVO - FORMANDO`, `CADASTRADO`, `MATRICULADO`, `TRANCADO` e variações |

> **Valor não reconhecido?** O código classifica como ATIVO por padrão (protocolo §4.1). Se sua instituição usa termos diferentes, adicione-os no dicionário `STATUS_MAP` na célula 3.

Conforme o protocolo (§4.1), registros sem status são considerados ATIVO; registros sem gênero ou ano de ingresso são excluídos. O filtro de nível de ensino mantém apenas registros de **graduação**.

In [12]:
def carregar_coorte(caminho):
    """Lê + normaliza + filtra TI + traduz status de um arquivo de coorte."""
    print(f"Lendo {os.path.basename(caminho)}...")
    df = ler_arquivo_qualquer(caminho)
    df = normalizar_colunas(df)
    if "curso" not in df.columns:
        raise ValueError(f"Coluna de curso não encontrada. Colunas: {list(df.columns)[:8]}")
    df = df[df["curso"].apply(lambda v: eh_ti(v) if pd.notna(v) else False)].copy()
    print(f" {len(df):,} registros em cursos de TI")

    # Normalização para o schema canônico
    g = df.get("genero", pd.Series(dtype=str)).astype(str).str.strip().str.upper()
    df["genero"] = g.map({"F": "F", "FEMININO": "F", "M": "M", "MASCULINO": "M"})
    df["ano_ingresso"] = pd.to_numeric(df.get("ano_ingresso", pd.Series(dtype=str)),
                                       errors="coerce").astype("Int64")
    if "ano_conclusao" in df.columns:
        df["ano_conclusao"] = pd.to_numeric(df["ano_conclusao"], errors="coerce").astype("Int64")
    df["status_canonico"] = normalizar_status(df["status"]) if "status" in df.columns else "ATIVO"
    df["curso"] = df["curso"].astype(str).str.strip().str.upper()

    # Foco em graduação (protocolo é sobre cursos de graduação em TI)
    if "nivel_ensino" in df.columns:
        ne = df["nivel_ensino"].astype(str).str.upper()
        df = df[ne.str.contains("GRADUA", na=False)]

    df = df.dropna(subset=["genero", "ano_ingresso"])
    return df


# Detecta automaticamente o(s) arquivo(s) COORTE dentro de PASTA
arquivos_coorte = []
for f in sorted(glob.glob(os.path.join(PASTA, "discentes-*.csv")) +
                glob.glob(os.path.join(PASTA, "discentes-*.xlsx")) +
                glob.glob(os.path.join(PASTA, "discentes-*.json"))):
    base = os.path.basename(f).lower()
    if "egress" not in base: # exclui discentes-egressos-*
        arquivos_coorte.append(f)

if not arquivos_coorte:
    print(" Nenhum arquivo COORTE encontrado (discentes-AAAA.csv sem 'egressos' no nome).")
    print(" Coloque pelo menos um arquivo como 'discentes-2019.csv' na pasta do Drive.")
    df_coorte = pd.DataFrame()
else:
    print(f"Arquivos COORTE detectados: {[os.path.basename(f) for f in arquivos_coorte]}\n")
    # Carrega e concatena todos os arquivos COORTE encontrados
    partes = []
    for f in arquivos_coorte:
        try:
            partes.append(carregar_coorte(f))
        except Exception as e:
            print(f" Erro ao ler {os.path.basename(f)}: {e}")
    df_coorte = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

if len(df_coorte):
    print(f"\nTotal após filtros (graduação TI): {len(df_coorte):,} registros")
    print("\nDistribuição por status canônico:")
    print(df_coorte["status_canonico"].value_counts())
    print("\nDistribuição por curso:")
    print(df_coorte["curso"].value_counts())

Arquivos COORTE detectados: ['discentes-2017.csv', 'discentes-2018.csv', 'discentes-2019.csv']

Lendo discentes-2017.csv...
 2,653 registros em cursos de TI
Lendo discentes-2018.csv...
 866 registros em cursos de TI
Lendo discentes-2019.csv...
 859 registros em cursos de TI

Total após filtros (graduação TI): 1,815 registros

Distribuição por status canônico:
status_canonico
EVADIDO       1017
CONCLUINTE     724
ATIVO           74
Name: count, dtype: int64

Distribuição por curso:
curso
TECNOLOGIA DA INFORMAÇÃO                 1021
ENGENHARIA DE COMPUTAÇÃO                  214
SISTEMAS DE INFORMAÇÃO                    145
ENGENHARIA MECATRÔNICA                    108
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS     105
ENGENHARIA DE TELECOMUNICAÇÕES             99
ENGENHARIA DE SOFTWARE                     71
CIÊNCIA DA COMPUTAÇÃO                      52
Name: count, dtype: int64


## 6. Calcular M1, M2 e M3 a partir da coorte

`calcular_metricas_coorte(df)` agrega por `ano_ingresso` e devolve, para cada coorte:

- `ingressantes_F`, `ingressantes_M`, `ingressantes_total` → numerador e denominador de **M1**
- `evadidos_F`, `evadidos_M` → numeradores de **TEF** e **TEM**
- `concluintes_F`, `concluintes_M` → numerador de **M3**
- `M1_perc_mulheres_ingressantes`, `TEF`, `TEM`, `M2_gap_evasao`, `M3_perc_mulheres_concluintes`

O **gap de evasão** é positivo quando mulheres evadem mais que homens (situação adversa). É exatamente `TEF − TEM` definido no protocolo.

In [14]:
def calcular_metricas_coorte(df, col_coorte="ano_ingresso"):
    """Calcula M1, M2 (TEF/TEM/gap) e M3 por coorte."""
    rows = []
    for ano in sorted(df[col_coorte].dropna().unique()):
        d = df[df[col_coorte] == ano]
        d_f = d[d["genero"] == "F"]
        d_m = d[d["genero"] == "M"]
        nF, nM = len(d_f), len(d_m)
        nT = nF + nM
        evF = (d_f["status_canonico"] == "EVADIDO").sum()
        evM = (d_m["status_canonico"] == "EVADIDO").sum()
        ccF = (d_f["status_canonico"] == "CONCLUINTE").sum()
        ccM = (d_m["status_canonico"] == "CONCLUINTE").sum()

        m1 = round(nF / nT * 100, 1) if nT else 0
        tef = round(evF / nF * 100, 1) if nF else 0
        tem = round(evM / nM * 100, 1) if nM else 0
        m2 = round(tef - tem, 1)
        ccT = ccF + ccM
        m3 = round(ccF / ccT * 100, 1) if ccT else 0
        rows.append(dict(
            coorte=int(ano),
            ingressantes_F=nF, ingressantes_M=nM, ingressantes_total=nT,
            evadidos_F=int(evF), evadidos_M=int(evM),
            concluintes_F=int(ccF), concluintes_M=int(ccM),
            M1_perc_mulheres_ingressantes=m1,
            TEF=tef, TEM=tem, M2_gap_evasao=m2,
            M3_perc_mulheres_concluintes=m3,
        ))
    return pd.DataFrame(rows)


metricas = calcular_metricas_coorte(df_coorte)
metricas

,coorte,ingressantes_F,ingressantes_M,ingressantes_total,evadidos_F,evadidos_M,concluintes_F,concluintes_M,M1_perc_mulheres_ingressantes,TEF,TEM,M2_gap_evasao,M3_perc_mulheres_concluintes
0,2017,59,529,588,32,324,27,196,10.0,54.2,61.2,-7.0,12.1
1,2018,90,521,611,39,290,49,214,14.7,43.3,55.7,-12.4,18.6
2,2019,93,523,616,43,289,45,193,15.1,46.2,55.3,-9.1,18.9


## 7. Fase crítica detectada automaticamente

A função `identificar_fase_critica(m1, m2, m3)` aplica os limiares da Tabela 2 do protocolo e devolve **a fase mais grave** com a recomendação correspondente (curto, médio e longo prazo).

| Fase | Condição de alerta | Gravidade |
|---|---|---|
| Ingresso (M1) | M1 < 30% | Crítica se < 20% · Significativa se 20-30% |
| Permanência (M2) | Gap > 5 p.p. desfavorável às mulheres | Crítica se > 15 p.p. · Moderada se 5-15 p.p. |
| Conclusão (M3) | M3 < M1 − 5 p.p. | Perda no pipeline |

In [15]:
def classif_m1(v):
 if v < 20: return "Disparidade Crítica", CORES["vermelho"]
 if v < 30: return "Disparidade Significativa", CORES["amarelo"]
 if v < 40: return "Desbalanceamento Moderado", CORES["amarelo"]
 if v < 50: return "Aproximação da Paridade", CORES["verde"]
 return "Paridade Alcançada", CORES["verde"]


def identificar_fase_critica(m1, m2=None, m3=None):
 """Devolve dict {fase, gravidade, alerta, recomendacao} ranqueando
 Ingresso/Permanência/Conclusão e elegendo a mais grave como crítica."""
 alertas = []
 if m1 is not None:
 if m1 < 20: alertas.append(("Ingresso", 3, f"M1={m1}% (<20%, Crítica)"))
 elif m1 < 30: alertas.append(("Ingresso", 2, f"M1={m1}% (20-30%, Significativa)"))
 if m2 is not None:
 if m2 > 15: alertas.append(("Permanência", 3, f"M2={m2} p.p. (>15, Crítica)"))
 elif m2 > 5: alertas.append(("Permanência", 2, f"M2={m2} p.p. (5-15, Moderada)"))
 if m1 is not None and m3 is not None:
 if m3 < m1 - 5:
 alertas.append(("Conclusão", 2, f"M3={m3}% < M1−5 = {round(m1-5,1)}% (perda no pipeline)"))
 if not alertas:
 return dict(fase="—", gravidade=0,
 alerta="Nenhum alerta acima do limiar do protocolo.",
 recomendacao="Manter monitoramento contínuo.")
 fase, gravidade, alerta = sorted(alertas, key=lambda x: -x[1])[0]
 rec_por_fase = {
 "Ingresso": (
 "Curto prazo: palestras em escolas e olimpíadas de programação para meninas. "
 "Médio prazo: mentoria reversa e dias de imersão de alunas do ensino médio no curso. "
 "Longo prazo: inserir computação no currículo do ensino médio e produzir conteúdo "
 "audiovisual desconstruindo estereótipos."),
 "Permanência": (
 "Curto prazo: tutoria entre pares e grupos de afinidade femininos. "
 "Médio prazo: bolsas e auxílio-creche; revisar carga horária e práticas docentes hostis. "
 "Longo prazo: estruturar política institucional de combate ao assédio e formar docentes "
 "em pedagogia inclusiva."),
 "Conclusão": (
 "Curto prazo: identificar gargalos curriculares e oferecer disciplinas de recuperação. "
 "Médio prazo: programas de mentoria com egressas e estágios direcionados. "
 "Longo prazo: parcerias com empresas para empregabilidade pós-curso e visibilidade "
 "de profissionais mulheres na área."),
 }
 return dict(fase=fase, gravidade=gravidade, alerta=alerta,
 recomendacao=rec_por_fase[fase])


ult = metricas.iloc[-1]
m1v = float(ult["M1_perc_mulheres_ingressantes"])
m2v = float(ult["M2_gap_evasao"])
m3v = float(ult["M3_perc_mulheres_concluintes"]) if (ult["concluintes_F"]+ult["concluintes_M"]) else None

fase = identificar_fase_critica(m1v, m2v, m3v)
cls_m1, _ = classif_m1(m1v)

print(f"Coorte de referência: {int(ult['coorte'])}")
print(f" M1 = {m1v:.1f}% → {cls_m1}")
print(f" M2 = {m2v:+.1f} p.p. (TEF={ult['TEF']}%, TEM={ult['TEM']}%)")
print(f" M3 = {m3v}% \n")
print(f"FASE CRÍTICA: {fase['fase']} (gravidade {fase['gravidade']})")
print(f" Alerta: {fase['alerta']}")
print(f" Recomendação:\n {fase['recomendacao']}")

IndentationError: expected an indented block after 'if' statement on line 13 (3978369651.py, line 14)

In [16]:
def classif_m1(v):
    if v < 20: return "Disparidade Crítica", CORES["vermelho"]
    if v < 30: return "Disparidade Significativa", CORES["amarelo"]
    if v < 40: return "Desbalanceamento Moderado", CORES["amarelo"]
    if v < 50: return "Aproximação da Paridade", CORES["verde"]
    return "Paridade Alcançada", CORES["verde"]


def identificar_fase_critica(m1, m2=None, m3=None):
    """Devolve dict {fase, gravidade, alerta, recomendacao} ranqueando
    Ingresso/Permanência/Conclusão e elegendo a mais grave como crítica."""
    alertas = []
    if m1 is not None:
        if m1 < 20: alertas.append(("Ingresso", 3, f"M1={m1}% (<20%, Crítica)"))
        elif m1 < 30: alertas.append(("Ingresso", 2, f"M1={m1}% (20-30%, Significativa)"))
    if m2 is not None:
        if m2 > 15: alertas.append(("Permanência", 3, f"M2={m2} p.p. (>15, Crítica)"))
        elif m2 > 5: alertas.append(("Permanência", 2, f"M2={m2} p.p. (5-15, Moderada)"))
    if m1 is not None and m3 is not None:
        if m3 < m1 - 5:
            alertas.append(("Conclusão", 2, f"M3={m3}% < M1−5 = {round(m1-5,1)}% (perda no pipeline)"))
    if not alertas:
        return dict(fase="—", gravidade=0,
                    alerta="Nenhum alerta acima do limiar do protocolo.",
                    recomendacao="Manter monitoramento contínuo.")
    fase, gravidade, alerta = sorted(alertas, key=lambda x: -x[1])[0]
    rec_por_fase = {
        "Ingresso": (
            "Curto prazo: palestras em escolas e olimpíadas de programação para meninas. "
            "Médio prazo: mentoria reversa e dias de imersão de alunas do ensino médio no curso. "
            "Longo prazo: inserir computação no currículo do ensino médio e produzir conteúdo "
            "audiovisual desconstruindo estereótipos."),
        "Permanência": (
            "Curto prazo: tutoria entre pares e grupos de afinidade femininos. "
            "Médio prazo: bolsas e auxílio-creche; revisar carga horária e práticas docentes hostis. "
            "Longo prazo: estruturar política institucional de combate ao assédio e formar docentes "
            "em pedagogia inclusiva."),
        "Conclusão": (
            "Curto prazo: identificar gargalos curriculares e oferecer disciplinas de recuperação. "
            "Médio prazo: programas de mentoria com egressas e estágios direcionados. "
            "Longo prazo: parcerias com empresas para empregabilidade pós-curso e visibilidade "
            "de profissionais mulheres na área."),
    }
    return dict(fase=fase, gravidade=gravidade, alerta=alerta,
                recomendacao=rec_por_fase[fase])


fig_tef_tem = go.Figure([
    go.Bar(x=metricas["coorte"], y=metricas["TEF"], name="TEF (mulheres)",
           marker_color=CORES["f"]),
    go.Bar(x=metricas["coorte"], y=metricas["TEM"], name="TEM (homens)",
           marker_color=CORES["m"]),
])
fig_tef_tem.update_layout(title="M2 — Taxas de Evasão por Gênero",
                          xaxis_title="Coorte (ano de ingresso)",
                          yaxis_title="% evadidos da coorte",
                          barmode="group", template="plotly_white", height=400)
fig_tef_tem.show()

gap_cores = [
    CORES["verde"] if abs(v) <= 5 else
    CORES["amarelo"] if abs(v) <= 15 else
    CORES["vermelho"]
    for v in metricas["M2_gap_evasao"]
]
fig_gap = go.Figure(go.Bar(x=metricas["coorte"], y=metricas["M2_gap_evasao"],
                           marker_color=gap_cores))
fig_gap.add_hline(y=0, line_color="black", line_width=1)
fig_gap.add_hline(y=5, line_dash="dash", line_color=CORES["amarelo"], annotation_text="+5 p.p.")
fig_gap.add_hline(y=-5, line_dash="dash", line_color=CORES["amarelo"], annotation_text="-5 p.p.")
fig_gap.update_layout(title="M2 — Gap (TEF − TEM)",
                      xaxis_title="Coorte", yaxis_title="p.p.",
                      template="plotly_white", height=380)
fig_gap.show()

fig_status = go.Figure()
for status, cor in [("CONCLUINTE", CORES["verde"]),
                    ("EVADIDO", CORES["vermelho"]),
                    ("ATIVO", CORES["amarelo"])]:
    sub = df_coorte[df_coorte["status_canonico"] == status]
    if len(sub):
        ag = sub.groupby(["ano_ingresso", "genero"]).size().unstack(fill_value=0)
        fig_status.add_trace(go.Bar(
            x=ag.index.astype(int), y=ag.get("F", 0),
            name=f"{status} ♀", marker_color=cor, opacity=0.95))
        fig_status.add_trace(go.Bar(
            x=ag.index.astype(int), y=ag.get("M", 0),
            name=f"{status} ♂", marker_color=cor, opacity=0.45))
fig_status.update_layout(title="Composição da coorte por status acadêmico",
                         xaxis_title="Coorte", yaxis_title="Pessoas",
                         barmode="group", template="plotly_white", height=420)
fig_status.show()


## 8. Visualizações finais

Três gráficos:

- **M2 — Taxas de evasão por gênero** (TEF vs TEM, lado a lado).
- **M2 — Gap (TEF − TEM)** em p.p., com cores ligadas às faixas do protocolo (verde ≤5, amarelo ≤15, vermelho >15).
- **Composição da coorte por status acadêmico** — quantos ATIVOS / EVADIDOS / CONCLUINTES por gênero e ano de ingresso.

In [18]:
fig_tef_tem = go.Figure([
    go.Bar(x=metricas["coorte"], y=metricas["TEF"], name="TEF (mulheres)",
           marker_color=CORES["f"]),
    go.Bar(x=metricas["coorte"], y=metricas["TEM"], name="TEM (homens)",
           marker_color=CORES["m"]),
])
fig_tef_tem.update_layout(title="M2 — Taxas de Evasão por Gênero",
                          xaxis_title="Coorte (ano de ingresso)",
                          yaxis_title="% evadidos da coorte",
                          barmode="group", template="plotly_white", height=400)
fig_tef_tem.show()

gap_cores = [
    CORES["verde"] if abs(v) <= 5 else
    CORES["amarelo"] if abs(v) <= 15 else
    CORES["vermelho"]
    for v in metricas["M2_gap_evasao"]
]
fig_gap = go.Figure(go.Bar(x=metricas["coorte"], y=metricas["M2_gap_evasao"],
                           marker_color=gap_cores))
fig_gap.add_hline(y=0, line_color="black", line_width=1)
fig_gap.add_hline(y=5, line_dash="dash", line_color=CORES["amarelo"], annotation_text="+5 p.p.")
fig_gap.add_hline(y=-5, line_dash="dash", line_color=CORES["amarelo"], annotation_text="-5 p.p.")
fig_gap.update_layout(title="M2 — Gap (TEF − TEM)",
                      xaxis_title="Coorte", yaxis_title="p.p.",
                      template="plotly_white", height=380)
fig_gap.show()

fig_status = go.Figure()
for status, cor in [("CONCLUINTE", CORES["verde"]),
                    ("EVADIDO", CORES["vermelho"]),
                    ("ATIVO", CORES["amarelo"])]:
    sub = df_coorte[df_coorte["status_canonico"] == status]
    if len(sub):
        ag = sub.groupby(["ano_ingresso", "genero"]).size().unstack(fill_value=0)
        fig_status.add_trace(go.Bar(
            x=ag.index.astype(int), y=ag.get("F", 0),
            name=f"{status} ♀", marker_color=cor, opacity=0.95))
        fig_status.add_trace(go.Bar(
            x=ag.index.astype(int), y=ag.get("M", 0),
            name=f"{status} ♂", marker_color=cor, opacity=0.45))
fig_status.update_layout(title="Composição da coorte por status acadêmico",
                         xaxis_title="Coorte", yaxis_title="Pessoas",
                         barmode="group", template="plotly_white", height=420)
fig_status.show()

## 10. Dashboard interativo completo

Monta o dashboard com Panel e o serve:

- **No Colab**: abre um link público temporário (via `pn.serve` + tunelamento do Colab). Clique no link que aparecer no output.
- **Localmente**: abre automaticamente em `http://localhost:<porta>` no seu navegador.

O dashboard contém:
- **Seção M1** — evolução anual do % de mulheres ingressantes, barras por curso e (se houver múltiplas IES) ranking institucional.
- **Seção M2 + M3** — gráficos de evasão por gênero, gap TEF−TEM e concluintes, construídos a partir dos arquivos COORTE/CONCLUSÃO.
- **Card de fase crítica** com recomendações automáticas.
- **Painel de termos TI** — edite as listas de inclusão/exclusão ao vivo e recalcule tudo sem reiniciar o notebook.

In [20]:
def _carregar_df_tipo(padrao_glob, tipo_esperado):
    """Carrega e concatena todos os arquivos de um dado tipo da pasta."""
    partes = []
    for f in sorted(glob.glob(padrao_glob)):
        try:
            df_tmp = ler_arquivo_qualquer(f)
            df_tmp = normalizar_colunas(df_tmp)
            if detectar_tipo_arquivo(df_tmp) == tipo_esperado:
                partes.append(df_tmp)
        except Exception:
            pass
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def construir_dashboard(df_coorte, pasta=PASTA):
    """Monta e retorna o layout Panel completo do dashboard."""
    # ── dados auxiliares da pasta ──────────────────────────────────────
    df_conclusao = _carregar_df_tipo(
        os.path.join(pasta, "discentes-egressos-*.csv"), "CONCLUSAO")
    df_ingresso = _carregar_df_tipo(
        os.path.join(pasta, "chamada_regular*.csv"), "INGRESSO")

    tem_coorte = len(df_coorte) > 0
    tem_conclusao = len(df_conclusao) > 0
    tem_ingresso = len(df_ingresso) > 0

    # ── widgets globais ────────────────────────────────────────────────
    inc_input = pn.widgets.TextAreaInput(
        name="Termos INCLUIR (1 por linha)",
        value="\n".join(CONFIG_TI["incluir"]), height=160,
        sizing_mode="stretch_width")
    exc_input = pn.widgets.TextAreaInput(
        name="Termos EXCLUIR (1 por linha)",
        value="\n".join(CONFIG_TI["excluir"]), height=160,
        sizing_mode="stretch_width")
    btn_ti = pn.widgets.Button(name="Aplicar e recalcular", button_type="primary")
    msg_ti = pn.pane.Markdown("")

    area_m1 = pn.Column(sizing_mode="stretch_width")
    area_m2m3 = pn.Column(sizing_mode="stretch_width")

    # ── M1 ─────────────────────────────────────────────────────────────
    def montar_m1():
        df_ing = df_ingresso.copy() if tem_ingresso else (
            df_coorte.copy() if tem_coorte else pd.DataFrame())
        if not len(df_ing) or "genero" not in df_ing.columns:
            area_m1.objects = [pn.pane.Alert(
                "Nenhum dado de ingresso carregado. Coloque um arquivo de ingressantes "
                "ou um arquivo COORTE na pasta.", alert_type="info")]
            return

        df_ing["genero"] = (df_ing["genero"].astype(str).str.strip().str.upper()
                            .map({"F":"F","FEMININO":"F","M":"M","MASCULINO":"M"}))
        col_ano = "ano_ingresso" if "ano_ingresso" in df_ing.columns else "ano"
        df_ing[col_ano] = pd.to_numeric(df_ing[col_ano], errors="coerce").astype("Int64")
        df_ing = df_ing.dropna(subset=["genero", col_ano])
        if "curso" in df_ing.columns:
            df_ing = df_ing[df_ing["curso"].apply(
                lambda v: eh_ti(v) if pd.notna(v) else False)]

        rows = []
        for ano in sorted(df_ing[col_ano].dropna().unique()):
            d = df_ing[df_ing[col_ano] == ano]
            t = len(d); f = (d["genero"] == "F").sum()
            rows.append(dict(ano=int(ano), total=t, mulheres=int(f),
                             homens=int(t-f), perc_f=round(f/t*100,1) if t else 0))
        m1_df = pd.DataFrame(rows)
        if not len(m1_df):
            area_m1.objects = [pn.pane.Alert("Sem dados M1 após filtros.", alert_type="warning")]
            return

        ultimo = float(m1_df["perc_f"].iloc[-1])
        cls, cor = classif_m1(ultimo)
        d_pp = round(ultimo - float(m1_df["perc_f"].iloc[0]), 1) if len(m1_df)>1 else 0
        sinal = "+" if d_pp > 0 else ""

        card = pn.pane.HTML(f'''
<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));
gap:10px;margin:10px 0">
<div style="background:{cor}15;border-left:4px solid {cor};
padding:12px;border-radius:6px">
<div style="color:#555;font-size:13px">M1 — % Mulheres Ingressantes</div>
<div style="font-size:26px;font-weight:700;color:{cor}">{ultimo}%
<span style="font-size:14px">({sinal}{d_pp} p.p.)</span></div>
<div style="color:#777;font-size:12px">{cls}</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="color:#555;font-size:13px">Ingressantes TI</div>
<div style="font-size:26px;font-weight:700">{len(df_ing):,}</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="color:#555;font-size:13px">Cursos TI</div>
<div style="font-size:26px;font-weight:700">
{df_ing["curso"].nunique() if "curso" in df_ing.columns else "—"}</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="color:#555;font-size:13px">Período</div>
<div style="font-size:26px;font-weight:700">
{int(m1_df["ano"].min())}–{int(m1_df["ano"].max())}</div>
</div>
</div>''', sizing_mode="stretch_width")

        fig_ev = go.Figure()
        fig_ev.add_trace(go.Scatter(x=m1_df["ano"], y=m1_df["perc_f"],
                                    mode="lines+markers", name="% Mulheres",
                                    line=dict(color=CORES["f"], width=3), marker=dict(size=8)))
        fig_ev.add_hline(y=30, line_dash="dash", line_color="gray",
                         annotation_text="Meta 30%")
        fig_ev.update_layout(title="M1 — % Mulheres Ingressantes em TI",
                             xaxis_title="Ano", yaxis_title="%",
                             yaxis_range=[0, max(55, m1_df["perc_f"].max()+10)],
                             template="plotly_white", height=380)

        fig_bar = go.Figure([
            go.Bar(x=m1_df["ano"], y=m1_df["mulheres"],
                   name="Mulheres", marker_color=CORES["f"]),
            go.Bar(x=m1_df["ano"], y=m1_df["homens"],
                   name="Homens", marker_color=CORES["m"]),
        ])
        fig_bar.update_layout(title="Ingressantes em TI por Gênero",
                              barmode="group", template="plotly_white", height=360)

        tabs_m1 = [
            ("Evolução", pn.Column(
                pn.pane.Plotly(fig_ev, sizing_mode="stretch_width"),
                pn.pane.Plotly(fig_bar, sizing_mode="stretch_width"),
                pn.pane.DataFrame(m1_df, sizing_mode="stretch_width"))),
        ]
        if "curso" in df_ing.columns:
            dd = []
            for c in df_ing["curso"].dropna().unique():
                s = df_ing[df_ing["curso"]==c]; t = len(s)
                dd.append(dict(curso=c,
                               perc=round((s["genero"]=="F").sum()/t*100,1) if t else 0, total=t))
            dc = pd.DataFrame(dd).sort_values("perc")
            fig_c = go.Figure(go.Bar(y=dc["curso"], x=dc["perc"], orientation="h",
                                     marker_color=CORES["f"],
                                     text=dc["perc"].apply(lambda v: f"{v}%"), textposition="outside"))
            fig_c.add_vline(x=30, line_dash="dash", line_color="gray",
                            annotation_text="Meta 30%")
            fig_c.update_layout(title="% Mulheres por Curso",
                                template="plotly_white",
                                height=max(350, len(dc)*28+80), margin=dict(l=300))
            tabs_m1.append(("Por Curso",
                            pn.pane.Plotly(fig_c, sizing_mode="stretch_width")))
        if "sigla_ies" in df_ing.columns:
            dd2 = []
            for ies in df_ing["sigla_ies"].dropna().unique():
                s = df_ing[df_ing["sigla_ies"]==ies]; t = len(s)
                if t < 30: continue
                dd2.append(dict(ies=ies,
                                perc=round((s["genero"]=="F").sum()/t*100,1), total=t))
            if dd2:
                dc2 = pd.DataFrame(dd2).sort_values("perc")
                fig_r = go.Figure(go.Bar(y=dc2["ies"], x=dc2["perc"], orientation="h",
                                         marker_color=CORES["verde"],
                                         text=dc2["perc"].apply(lambda v: f"{v}%"), textposition="outside"))
                fig_r.add_vline(x=30, line_dash="dash", line_color="gray")
                fig_r.update_layout(title="Ranking por Instituição (≥30 registros)",
                                    template="plotly_white", height=700, margin=dict(l=120))
                tabs_m1.append(("Ranking Instituições",
                                pn.pane.Plotly(fig_r, sizing_mode="stretch_width")))

        area_m1.objects = [card, pn.Tabs(*tabs_m1, dynamic=True)]

    # ── M2 + M3 ────────────────────────────────────────────────────────
    def montar_m2m3():
        if not tem_coorte and not tem_conclusao:
            area_m2m3.objects = [pn.pane.Alert(
                "Nenhum arquivo COORTE ou CONCLUSÃO encontrado. "
                "Para M2+M3, coloque um arquivo com coluna de situação acadêmica.",
                alert_type="info")]
            return

        conteudo = []

        # M2+M3 via COORTE
        if tem_coorte:
            met = calcular_metricas_coorte(df_coorte)
            ult = met.iloc[-1]
            m1v = float(ult["M1_perc_mulheres_ingressantes"])
            m2v = float(ult["M2_gap_evasao"])
            m3v = float(ult["M3_perc_mulheres_concluintes"]) \
                if (ult["concluintes_F"]+ult["concluintes_M"]) else None
            fase = identificar_fase_critica(m1v, m2v, m3v)
            cor_f = {3:CORES["vermelho"],2:CORES["amarelo"],0:CORES["verde"]}[fase["gravidade"]]
            _, cor_m1c = classif_m1(m1v)

            card_c = pn.pane.HTML(f'''
<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));
gap:10px;margin:10px 0">
<div style="background:{cor_m1c}15;border-left:4px solid {cor_m1c};
padding:12px;border-radius:6px">
<div style="font-size:12px;color:#555">M1 — % Mulheres Ingr.</div>
<div style="font-size:24px;font-weight:700;color:{cor_m1c}">{m1v}%</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="font-size:12px;color:#555">M2 — Gap Evasão (TEF−TEM)</div>
<div style="font-size:24px;font-weight:700">{m2v:+.1f} p.p.</div>
<div style="font-size:11px;color:#777">TEF={ult["TEF"]}% · TEM={ult["TEM"]}%</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="font-size:12px;color:#555">M3 — % Mulheres Conc.</div>
<div style="font-size:24px;font-weight:700">
{m3v if m3v is not None else "—"}{"%" if m3v is not None else ""}</div>
</div>
<div style="background:#f5f5f5;padding:12px;border-radius:6px">
<div style="font-size:12px;color:#555">Ingressantes</div>
<div style="font-size:24px;font-weight:700">{int(met["ingressantes_total"].sum()):,}</div>
</div>
</div>
<div style="background:{cor_f}15;border-left:5px solid {cor_f};
padding:10px 14px;border-radius:6px">
<b>Fase crítica:</b> {fase["fase"]} — {fase["alerta"]}<br>
<b>Recomendação:</b> {fase["recomendacao"]}
</div>''', sizing_mode="stretch_width")

            gc = [CORES["verde"] if abs(v)<=5 else CORES["amarelo"] if abs(v)<=15
                  else CORES["vermelho"] for v in met["M2_gap_evasao"]]
            fig_tef = go.Figure([
                go.Bar(x=met["coorte"], y=met["TEF"], name="TEF ♀",
                       marker_color=CORES["f"]),
                go.Bar(x=met["coorte"], y=met["TEM"], name="TEM ♂",
                       marker_color=CORES["m"]),
            ])
            fig_tef.update_layout(title="M2 — Taxas de Evasão por Gênero",
                                  xaxis_title="Coorte", yaxis_title="% evadidos",
                                  barmode="group", template="plotly_white", height=360)
            fig_gap = go.Figure(go.Bar(x=met["coorte"],
                                       y=met["M2_gap_evasao"], marker_color=gc))
            fig_gap.add_hline(y=0, line_color="black", line_width=1)
            fig_gap.add_hline(y=5, line_dash="dash", line_color=CORES["amarelo"],
                              annotation_text="+5 p.p.")
            fig_gap.add_hline(y=-5, line_dash="dash", line_color=CORES["amarelo"],
                              annotation_text="-5 p.p.")
            fig_gap.update_layout(title="M2 — Gap (TEF − TEM)",
                                  xaxis_title="Coorte", yaxis_title="p.p.",
                                  template="plotly_white", height=360)

            conteudo.append(pn.pane.HTML(
                '<h3 style="margin:16px 0 4px">M1 + M2 + M3 — Arquivo COORTE</h3>',
                sizing_mode="stretch_width"))
            conteudo.append(card_c)
            conteudo.append(pn.Tabs(
                ("M2 — Evasão", pn.Column(
                    pn.pane.Plotly(fig_tef, sizing_mode="stretch_width"),
                    pn.pane.Plotly(fig_gap, sizing_mode="stretch_width"))),
                ("Tabela completa",
                 pn.pane.DataFrame(met, sizing_mode="stretch_width")),
                dynamic=True))

        # M3 via CONCLUSÃO (egressos)
        if tem_conclusao:
            df_eg = df_conclusao.copy()
            df_eg["genero"] = (df_eg.get("genero", pd.Series(dtype=str))
                               .astype(str).str.strip().str.upper()
                               .map({"F":"F","FEMININO":"F","M":"M","MASCULINO":"M"}))
            col_eg = "ano_ingresso" if "ano_ingresso" in df_eg.columns else "ano_conclusao"
            df_eg[col_eg] = pd.to_numeric(df_eg[col_eg], errors="coerce").astype("Int64")
            if "nivel_ensino" in df_eg.columns:
                df_eg = df_eg[df_eg["nivel_ensino"].astype(str).str.upper()
                              .str.contains("GRADUA", na=False)]
            if "curso" in df_eg.columns:
                df_eg = df_eg[df_eg["curso"].apply(
                    lambda v: eh_ti(v) if pd.notna(v) else False)]
            df_eg = df_eg.dropna(subset=["genero", col_eg])

            rows3 = []
            for ano in sorted(df_eg[col_eg].dropna().unique()):
                d = df_eg[df_eg[col_eg]==ano]
                t = len(d); f = (d["genero"]=="F").sum()
                rows3.append(dict(ano=int(ano), total=t, mulheres=int(f),
                                  homens=int(t-f), perc_f=round(f/t*100,1) if t else 0))
            m3_df = pd.DataFrame(rows3)
            if len(m3_df):
                fig_m3 = go.Figure()
                fig_m3.add_trace(go.Scatter(
                    x=m3_df["ano"], y=m3_df["perc_f"], mode="lines+markers",
                    name="% Mulheres Concluintes",
                    line=dict(color=CORES["roxo"], width=3), marker=dict(size=8)))
                fig_m3.add_hline(y=30, line_dash="dash", line_color="gray",
                                 annotation_text="Meta 30%")
                fig_m3.update_layout(
                    title=f"M3 — % Mulheres Concluintes (eixo: {col_eg})",
                    xaxis_title=col_eg, yaxis_title="%",
                    yaxis_range=[0, max(55, m3_df["perc_f"].max()+10)],
                    template="plotly_white", height=380)
                conteudo.append(pn.pane.HTML(
                    '<h3 style="margin:24px 0 4px">M3 — Arquivo CONCLUSÃO (egressos)</h3>',
                    sizing_mode="stretch_width"))
                conteudo.append(pn.Tabs(
                    ("M3 — Evolução",
                     pn.pane.Plotly(fig_m3, sizing_mode="stretch_width")),
                    ("Tabela",
                     pn.pane.DataFrame(m3_df, sizing_mode="stretch_width")),
                    dynamic=True))

        area_m2m3.objects = conteudo or [pn.pane.Alert(
            "Nenhum dado processado.", alert_type="warning")]

    # ── Callback configuração TI ───────────────────────────────────────
    def aplicar_ti(event=None):
        CONFIG_TI["incluir"] = [l.strip().lower()
                                for l in inc_input.value.splitlines() if l.strip()]
        CONFIG_TI["excluir"] = [l.strip().lower()
                                for l in exc_input.value.splitlines() if l.strip()]
        montar_m1(); montar_m2m3()
        msg_ti.object = (f" Aplicado: **{len(CONFIG_TI['incluir'])}** termos incluir · "
                          f"**{len(CONFIG_TI['excluir'])}** excluir. Painéis atualizados.")
    btn_ti.on_click(aplicar_ti)

    montar_m1()
    montar_m2m3()

    # ── Layout ─────────────────────────────────────────────────────────
    return pn.Column(
        pn.pane.HTML('''
<div style="background:linear-gradient(135deg,#E91E63,#9C27B0);
padding:20px 28px;border-radius:10px;color:white;margin-bottom:12px">
<h1 style="margin:0;font-size:1.6em">Equidade de Gênero em TI</h1>
<p style="margin:4px 0 0;opacity:.85">
Protocolo Natal/2026 — M1 (ingresso) · M2 (evasão) · M3 (conclusão)</p>
</div>''', sizing_mode="stretch_width"),

        pn.pane.HTML('<h2 style="border-bottom:3px solid #1976D2;'
                     'padding-bottom:6px;margin-top:16px"> M1 — Ingressantes</h2>',
                     sizing_mode="stretch_width"),
        area_m1,

        pn.pane.HTML('<h2 style="border-bottom:3px solid #9C27B0;'
                     'padding-bottom:6px;margin-top:28px"> M2 + M3 — Evasão e Conclusão</h2>',
                     sizing_mode="stretch_width"),
        area_m2m3,

        pn.pane.HTML('<h2 style="border-bottom:3px solid #FFB300;'
                     'padding-bottom:6px;margin-top:28px"> Configuração de Termos TI</h2>',
                     sizing_mode="stretch_width"),
        pn.pane.Markdown(
            "Edite as listas abaixo e clique em **Aplicar** — todos os gráficos são recalculados."),
        pn.Row(inc_input, exc_input, sizing_mode="stretch_width"),
        btn_ti, msg_ti,

        max_width=1200, sizing_mode="stretch_width"
    )


# pn.extension precisa ser chamado POR SESSÃO do servidor,
# por isso passamos uma função (não o objeto) para pn.serve.
def criar_app():
    pn.extension('plotly', sizing_mode='stretch_width', template='fast')
    return construir_dashboard(df_coorte, pasta=PASTA)


# ── Serve o dashboard ──────────────────────────────────────────────────
PORT = 5006
try:
    # ── Google Colab ──────────────────────────────────────────────────
    from google.colab.output import eval_js
    pn.serve({"dashboard": criar_app}, port=PORT, show=False,
             threaded=True, websocket_origin="*")
    import time; time.sleep(1) # aguarda o servidor subir
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f" Abra o dashboard em:")
    print(f" {url}")
    print(" (link válido enquanto a sessão do Colab estiver ativa)")
except Exception as _e:
    # ── Ambiente local ────────────────────────────────────────────────
    pn.serve({"dashboard": criar_app}, port=PORT, show=True,
             threaded=True, websocket_origin="*")
    print(f" Dashboard em: http://localhost:{PORT}/dashboard")
    print(" Interrompa a célula (■) para encerrar o servidor.")

Launching server at http://localhost:5006
 Abra o dashboard em:
 https://5006-m-s-kkb-usw1b1-1dx3wut9ccqbh-b.us-west1-1.prod.colab.dev
 (link válido enquanto a sessão do Colab estiver ativa)


---

## Como adaptar este notebook para qualquer instituição

### Passo a passo completo

1. **Crie uma pasta no Google Drive** — ex.: `equidade_ti` (ou o nome que preferir).
2. **Coloque este notebook** (`dashboard_equidade_ti.ipynb`) dentro dessa pasta.
3. **Exporte os dados da sua instituição** e coloque os arquivos na mesma pasta. O notebook aceita dados de qualquer sistema acadêmico (SIGAA, SIG, INEP, SISU, etc.) nos formatos **CSV, Excel ou JSON**.
4. **Abra o notebook no Colab** (no Drive: botão direito → Abrir com → Google Colaboratory).
5. Se o nome da sua pasta for diferente de `equidade_ti`, edite `NOME_PASTA_DRIVE` na **célula 1 (Setup)**.
6. Execute todas as células (**Ambiente de execução → Executar tudo**).
 - Na primeira execução o Colab pedirá permissão para acessar o Drive — clique em **Permitir**.

---

### O que fazer se o arquivo não for reconhecido corretamente

**Coluna de gênero com nome diferente?**
Adicione o nome na lista `genero` dentro de `SINONIMOS_COLUNAS` (célula 3):
```python
"genero": ["sexo", "tp_sexo", "genero", "gen_aluno", "SEU_NOME_AQUI"],
```

**Coluna de situação com nome diferente?**
Adicione na lista `status` dentro de `SINONIMOS_COLUNAS`:
```python
"status": ["status", "situacao", "situacao_vinculo", "SEU_NOME_AQUI"],
```

**Valor de situação não reconhecido?** (ex.: sua instituição usa `DESISTÊNCIA` em vez de `CANCELADO`)
Adicione no dicionário `STATUS_MAP` (célula 3):
```python
"DESISTÊNCIA": "EVADIDO",
```

**Curso de TI não detectado ou curso errado sendo incluído?**
Edite as listas em `CONFIG_TI` (célula 2):
```python
# Para incluir um novo curso:
"incluir": [..., "biomedica"],

# Para excluir um falso positivo:
"excluir": [..., "termo_indesejado"],
```

---

## Limitações reconhecidas (alinhadas ao §6 do protocolo)

- **Gênero como sexo biológico** (M/F) — não captura identidades não-binárias ou transgêneras.
- **Coortes recentes** (ingresso há menos de 5 anos) ainda estão em curso; M3 dessas coortes é parcial e deve ser interpretado como tal.
- **Arquivos só de ingressantes** (sem coluna de situação) → M2 não pode ser calculado; é necessário um arquivo que inclua a situação acadêmica de cada aluno.